# 第2回: PyTorch の Layer 理解

この notebook では、PyTorch の主要なレイヤーについて学びます。

- **BatchNormalization**: バッチ方向の正規化
- **LayerNormalization**: レイヤー方向の正規化
- **SpatialSoftmax / InverseSpatialSoftmax**: 特徴マップから座標への変換
- **Transformer**: Self-Attention とシーケンスモデリング

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False


## 1. BatchNormalization

`nn.BatchNorm1d` はバッチ方向（サンプル間）に平均0・分散1へ正規化します。
学習時はミニバッチの統計量を使い、推論時は移動平均を使います。

In [ ]:
torch.manual_seed(42)

batch_size = 8
features = 4

# ランダムデータ（平均・分散がバラバラ）
x = torch.randn(batch_size, features) * torch.tensor([5.0, 0.1, 3.0, 10.0]) + torch.tensor([10.0, -5.0, 0.0, 20.0])
print("入力データの形状:", x.shape)
print("入力データ（バッチ方向）の平均:", x.mean(dim=0))
print("入力データ（バッチ方向）の分散:", x.var(dim=0))

In [ ]:
bn = nn.BatchNorm1d(features)
bn.train()

y = bn(x)
print("正規化後（バッチ方向）の平均:", y.mean(dim=0))
print("正規化後（バッチ方向）の分散:", y.var(dim=0, unbiased=False))

正規化後、各特徴量のバッチ方向の平均 ≈ 0、分散 ≈ 1 になっていることを確認できます。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 正規化前
axes[0].bar(range(features), x.mean(dim=0).detach().numpy(), yerr=x.std(dim=0).detach().numpy(), capsize=5)
axes[0].set_title("正規化前: バッチ方向の平均 ± 標準偏差")
axes[0].set_xlabel("特徴量のインデックス")
axes[0].set_ylabel("値")
axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5)

# 正規化後
axes[1].bar(range(features), y.mean(dim=0).detach().numpy(), yerr=y.std(dim=0).detach().numpy(), capsize=5, color='orange')
axes[1].set_title("正規化後: バッチ方向の平均 ± 標準偏差")
axes[1].set_xlabel("特徴量のインデックス")
axes[1].set_ylabel("値")
axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 2. LayerNormalization

`nn.LayerNorm` はレイヤー方向（各サンプル内の特徴量方向）に正規化します。
BatchNorm はバッチサイズに依存しますが、LayerNorm はバッチサイズに依存しないため、
シーケンスモデルや小バッチ学習で広く使われます。

In [ ]:
ln = nn.LayerNorm(features)

y_ln = ln(x)
print("LayerNorm 後（レイヤー方向）の平均:", y_ln.mean(dim=-1))
print("LayerNorm 後（レイヤー方向）の分散:", y_ln.var(dim=-1, unbiased=False))

### BatchNorm vs LayerNorm の比較

| 特徴 | BatchNorm | LayerNorm |
|------|-----------|-----------|
| 正規化の方向 | バッチ方向（サンプル間） | レイヤー方向（特徴量間） |
| 計算単位 | 各特徴量ごとにバッチ統計量 | 各サンプルごとに特徴量統計量 |
| バッチサイズ依存 | あり（小バッチで不安定） | なし |
| 推論時の挙動 | 移動平均を使用 | 入力から直接計算 |
| 主な用途 | CNN、MLP | Transformer、RNN |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BatchNorm: バッチ方向の正規化を可視化
bn_out = bn(x).detach().numpy()
im0 = axes[0].imshow(bn_out, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
axes[0].set_title("BatchNorm: バッチ方向に正規化")
axes[0].set_xlabel("特徴量")
axes[0].set_ylabel("サンプル")
plt.colorbar(im0, ax=axes[0])

# LayerNorm: レイヤー方向の正規化を可視化
ln_out = ln(x).detach().numpy()
im1 = axes[1].imshow(ln_out, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
axes[1].set_title("LayerNorm: レイヤー方向に正規化")
axes[1].set_xlabel("特徴量")
axes[1].set_ylabel("サンプル")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

## 3. SpatialSoftmax & InverseSpatialSoftmax

**SpatialSoftmax** は CNN の特徴マップから各チャネルの「注目位置」を XY 座標として抽出する手法です。
ロボティクスの視覚入力処理で広く使われています。

**InverseSpatialSoftmax** は逆に、XY 座標からヒートマップ（ガウシアン）を生成します。

In [ ]:
class SpatialSoftmax(nn.Module):
    def __init__(self, width, height, temperature=1e-4):
        super().__init__()
        self.temperature = temperature
        pos_x, pos_y = np.meshgrid(
            np.linspace(0, 1, height), np.linspace(0, 1, width), indexing="xy"
        )
        self.register_buffer("pos_x", torch.from_numpy(pos_x.reshape(-1)).float())
        self.register_buffer("pos_y", torch.from_numpy(pos_y.reshape(-1)).float())

    def forward(self, x):
        B, C, W, H = x.shape
        logit = x.reshape(B, C, -1)
        att = torch.softmax(logit / self.temperature, dim=-1)
        ex = torch.sum(self.pos_x * att, dim=-1, keepdim=True)
        ey = torch.sum(self.pos_y * att, dim=-1, keepdim=True)
        keys = torch.cat([ex, ey], -1).reshape(B, C, 2)
        return keys, att.reshape(B, C, W, H)


class InverseSpatialSoftmax(nn.Module):
    def __init__(self, width, height, heatmap_size=0.1):
        super().__init__()
        self.heatmap_size = heatmap_size
        pos_x, pos_y = np.meshgrid(
            np.linspace(0, 1, height), np.linspace(0, 1, width), indexing="xy"
        )
        self.register_buffer(
            "pos_xy", torch.from_numpy(np.stack([pos_x, pos_y], axis=0)).float()
        )

    def forward(self, keys):
        d = torch.sum(
            torch.pow(self.pos_xy[None, None] - keys[:, :, :, None, None], 2.0), axis=2
        )
        return torch.exp(-d / self.heatmap_size)


print("SpatialSoftmax と InverseSpatialSoftmax を定義しました。")

In [ ]:
# ランダム特徴マップで実験
torch.manual_seed(0)
W, H = 16, 16
C = 3  # 3チャネル
B = 1

# ランダム特徴マップを作成（各チャネルに異なるピーク位置を持たせる）
feature_map = torch.zeros(B, C, W, H)
for c in range(C):
    cx, cy = np.random.randint(2, W - 2), np.random.randint(2, H - 2)
    feature_map[0, c, cx - 1:cx + 2, cy - 1:cy + 2] = 5.0
    feature_map[0, c] += torch.randn(W, H) * 0.1

# SpatialSoftmax で座標を抽出
ssm = SpatialSoftmax(W, H)
keys, attention = ssm(feature_map)
print("抽出された座標 (チャネルごとの XY):")
for c in range(C):
    print(f"  チャネル {c}: x={keys[0, c, 0]:.3f}, y={keys[0, c, 1]:.3f}")

# InverseSpatialSoftmax でヒートマップを再構成
issm = InverseSpatialSoftmax(W, H)
heatmaps = issm(keys)
print(f"再構成ヒートマップの形状: {heatmaps.shape}")

In [ ]:
fig, axes = plt.subplots(2, C, figsize=(4 * C, 8))
channel_names = ["チャネル 0", "チャネル 1", "チャネル 2"]

for c in range(C):
    # 元の特徴マップ + 抽出座標
    axes[0, c].imshow(feature_map[0, c].numpy(), cmap='viridis')
    # 座標をプロット（正規化座標をピクセル座標に変換）
    kx = keys[0, c, 0].item() * (H - 1)
    ky = keys[0, c, 1].item() * (W - 1)
    axes[0, c].plot(kx, ky, 'r+', markersize=15, markeredgewidth=3)
    axes[0, c].set_title(f"元の特徴マップ ({channel_names[c]})")

    # 再構成ヒートマップ
    axes[1, c].imshow(heatmaps[0, c].detach().numpy(), cmap='hot')
    axes[1, c].set_title(f"再構成ヒートマップ ({channel_names[c]})")

plt.suptitle("SpatialSoftmax → 座標抽出 → InverseSpatialSoftmax → ヒートマップ再構成", y=1.02)
plt.tight_layout()
plt.show()

## 4. Transformer

Transformer は **Self-Attention** メカニズムに基づくアーキテクチャです。

### Self-Attention の概念
入力シーケンスの各要素が、他のすべての要素との関連度を計算し、
重み付き和を出力します。

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Multi-Head Attention
複数の Attention ヘッドを並列に計算し、異なる「観点」から情報を集約します。

In [ ]:
# nn.TransformerEncoderLayer の基本的な使い方
d_model = 32   # 特徴量の次元
nhead = 4      # Attention ヘッド数
seq_len = 10   # シーケンス長
batch_size = 2

encoder_layer = nn.TransformerEncoderLayer(
    d_model=d_model, nhead=nhead, dim_feedforward=64, dropout=0.1, batch_first=True
)

x_seq = torch.randn(batch_size, seq_len, d_model)
out = encoder_layer(x_seq)
print(f"入力の形状:  {x_seq.shape}")
print(f"出力の形状:  {out.shape}")
print("→ 入力と同じ形状の出力が得られます（各位置の表現が更新されます）")

### 簡単な時系列予測デモ

sin 波を入力として、次のステップの値を予測する簡単な Transformer モデルを構築します。

In [ ]:
class SimpleTransformerPredictor(nn.Module):
    def __init__(self, d_model=32, nhead=4, num_layers=2):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=64,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: (batch, seq_len, 1)
        h = self.input_proj(x)
        h = self.transformer(h)
        return self.output_proj(h)

model = SimpleTransformerPredictor()
print(model)

In [ ]:
# 学習データの生成: sin 波
t = np.linspace(0, 8 * np.pi, 200)
signal = np.sin(t)

# スライディングウィンドウでデータセットを作成
window = 20
X_list, Y_list = [], []
for i in range(len(signal) - window):
    X_list.append(signal[i:i + window])
    Y_list.append(signal[i + 1:i + window + 1])

X_train = torch.tensor(np.array(X_list), dtype=torch.float32).unsqueeze(-1)
Y_train = torch.tensor(np.array(Y_list), dtype=torch.float32).unsqueeze(-1)
print(f"学習データの形状: X={X_train.shape}, Y={Y_train.shape}")

In [ ]:
# 学習
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

losses = []
model.train()
for epoch in range(100):
    pred = model(X_train)
    loss = criterion(pred, Y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}, Loss: {loss.item():.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 学習損失
axes[0].plot(losses)
axes[0].set_xlabel("エポック")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("学習損失の推移")
axes[0].set_yscale("log")

# 予測結果
model.eval()
with torch.no_grad():
    test_input = X_train[:1]  # 最初のウィンドウ
    pred = model(test_input)

axes[1].plot(range(window), test_input[0, :, 0].numpy(), 'b-o', label="入力", markersize=3)
axes[1].plot(range(window), pred[0, :, 0].numpy(), 'r--x', label="予測", markersize=3)
axes[1].plot(range(window), Y_train[0, :, 0].numpy(), 'g:', label="正解", markersize=3)
axes[1].legend()
axes[1].set_title("Transformer による時系列予測")
axes[1].set_xlabel("ステップ")
axes[1].set_ylabel("値")

plt.tight_layout()
plt.show()

## 演習問題

### 演習1: BatchNorm2d を画像データに適用

ランダムな画像データ（バッチ=4, チャネル=3, 高さ=8, 幅=8）に `nn.BatchNorm2d` を適用し、
正規化前後のチャネルごとの平均・分散を比較して棒グラフで可視化してください。

In [ ]:
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
torch.manual_seed(42)
images = torch.randn(4, 3, 8, 8) * torch.tensor([5.0, 0.5, 2.0]).view(1, 3, 1, 1) + \
         torch.tensor([10.0, -3.0, 0.0]).view(1, 3, 1, 1)

bn2d = nn.BatchNorm2d(3)
bn2d.train()
out = bn2d(images)

# チャネルごとの統計量
mean_before = images.mean(dim=(0, 2, 3)).detach().numpy()
var_before = images.var(dim=(0, 2, 3)).detach().numpy()
mean_after = out.mean(dim=(0, 2, 3)).detach().numpy()
var_after = out.var(dim=(0, 2, 3), unbiased=False).detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ch = ["R", "G", "B"]
axes[0].bar(ch, mean_before, yerr=np.sqrt(var_before), capsize=5)
axes[0].set_title("正規化前: チャネルごとの平均")
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].bar(ch, mean_after, yerr=np.sqrt(var_after), capsize=5, color='orange')
axes[1].set_title("正規化後: チャネルごとの平均")
axes[1].axhline(y=0, color='r', linestyle='--')
plt.tight_layout()
plt.show()
```

</details>

### 演習2: SpatialSoftmax で注意点を抽出

上で定義した `SpatialSoftmax` クラスを使い、ランダムな特徴マップ（バッチ=1, チャネル=4, 幅=32, 高さ=32）から
座標を抽出し、元の特徴マップ上に抽出した座標をプロットしてください。

In [ ]:
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
torch.manual_seed(123)
W, H = 32, 32
C = 4
fmap = torch.randn(1, C, W, H)
# 各チャネルにピークを追加
for c in range(C):
    px, py = np.random.randint(4, W-4), np.random.randint(4, H-4)
    fmap[0, c, px-2:px+3, py-2:py+3] += 10.0

ssm = SpatialSoftmax(W, H)
keys, att = ssm(fmap)

fig, axes = plt.subplots(1, C, figsize=(4*C, 4))
for c in range(C):
    axes[c].imshow(fmap[0, c].numpy(), cmap='viridis')
    kx = keys[0, c, 0].item() * (H - 1)
    ky = keys[0, c, 1].item() * (W - 1)
    axes[c].plot(kx, ky, 'r+', markersize=15, markeredgewidth=3)
    axes[c].set_title(f"チャネル {c}")
plt.suptitle("SpatialSoftmax: 抽出された注意座標")
plt.tight_layout()
plt.show()
```

</details>

### 演習3: TransformerEncoder で系列予測モデルを構築

cos 波のデータを使って、`nn.TransformerEncoder` ベースの予測モデルを構築・学習し、
予測結果を可視化してください。ヒント: 上記の `SimpleTransformerPredictor` を参考に改良してみましょう。

In [ ]:
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
# cos 波データの生成
t = np.linspace(0, 6 * np.pi, 150)
signal = np.cos(t)

window = 15
X_list, Y_list = [], []
for i in range(len(signal) - window):
    X_list.append(signal[i:i+window])
    Y_list.append(signal[i+1:i+window+1])

X = torch.tensor(np.array(X_list), dtype=torch.float32).unsqueeze(-1)
Y = torch.tensor(np.array(Y_list), dtype=torch.float32).unsqueeze(-1)

# モデル構築
model2 = SimpleTransformerPredictor(d_model=16, nhead=4, num_layers=3)
optimizer = torch.optim.Adam(model2.parameters(), lr=1e-3)
criterion = nn.MSELoss()

model2.train()
for epoch in range(150):
    pred = model2(X)
    loss = criterion(pred, Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.6f}")

model2.eval()
with torch.no_grad():
    pred = model2(X[:1])

plt.figure(figsize=(10, 4))
plt.plot(X[0,:,0].numpy(), 'b-o', label="入力", markersize=3)
plt.plot(pred[0,:,0].numpy(), 'r--x', label="予測", markersize=3)
plt.plot(Y[0,:,0].numpy(), 'g:', label="正解", markersize=3)
plt.legend()
plt.title("Transformer による cos 波予測")
plt.xlabel("ステップ")
plt.ylabel("値")
plt.show()
```

</details>